# Módulo 12 — Despliegue de un modelo KNN en Vercel
### ¿Lloverá mañana? Ahora preguntándole a los días más parecidos

Ya desplegamos una **regresión lineal** (vivienda) y una **regresión logística** (lluvia). Ahora desplegamos **KNN** (*k vecinos más cercanos*) con los mismos días de lluvia del Módulo 8.

KNN tiene algo especial: **no aprende pesos**. Para clasificar un día nuevo busca los *k* días del entrenamiento que más se le parecen y deja que voten. Eso cambia por completo lo que hay que desplegar:

| | Regresión lineal | Regresión logística | **KNN** |
|---|---|---|---|
| ¿Qué aprende al entrenar? | Pesos y sesgo | Pesos y sesgo | **Nada: solo guarda los datos** |
| ¿Qué guarda el JSON? | μ, σ, pesos, sesgo | pesos, sesgo, umbral | **μ, σ, k y todos los días de entrenamiento** |
| ¿Qué hace al predecir? | Una suma | Una suma y una sigmoide | **Calcula la distancia a cada día guardado** |
| ¿Crece el modelo con los datos? | No | No | **Sí** |

**Contenido:**
1. Los datos (los mismos del Módulo 8)
2. Elegir k y entrenar
3. ¿Qué es un modelo KNN? Exportarlo a `modelo.json`
4. Verificar que el JSON clasifica igual que scikit-learn
5. Construir la app
6. Probarla antes de desplegar
7. Descargar los archivos
8. Desplegar en Vercel
9. Retos

## 0. Preparar el entorno

In [ ]:
!pip install -q fastapi uvicorn httpx

import json
import math
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, ConfusionMatrixDisplay

print("Listo ✔")

## 1. Los datos: los mismos días del Módulo 8

Misma semilla, mismos 300 días. Así podremos comparar KNN con la regresión logística en igualdad de condiciones.

In [ ]:
np.random.seed(42)
n = 300

temperatura = np.random.normal(24, 5, n)
humedad = np.random.normal(65, 15, n)
viento = np.random.normal(10, 5, n)

z = -6 + 0.09 * humedad - 0.05 * temperatura + 0.02 * viento
prob_real = 1 / (1 + np.exp(-z))
llovio = (np.random.rand(n) < prob_real).astype(int)

df = pd.DataFrame({
    "temperatura": temperatura.round(1),
    "humedad": humedad.round(1),
    "viento": viento.round(1),
    "llovio": llovio,
})

variables = ["temperatura", "humedad", "viento"]
X = df[variables]
y = df["llovio"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Entrenamiento: {len(X_train)} días | Prueba: {len(X_test)} días")

## 2. Normalizar, elegir k y entrenar

### ¿Por qué aquí sí hay que normalizar?

KNN decide por **distancias**. Si no normalizamos, la humedad (que va de 20 a 100) pesa mucho más en la distancia que el viento (de 0 a 25), solo por su escala. Normalizando, cada variable aporta en igualdad de condiciones.

Como en la regresión lineal, las medias y desviaciones se calculan **solo con los datos de entrenamiento** y habrá que guardarlas en el JSON.

In [ ]:
media = X_train.mean().values
desviacion = X_train.std().values

Xs_train = (X_train.values - media) / desviacion
Xs_test = (X_test.values - media) / desviacion

### ¿Cuántos vecinos?

Probamos varios valores de *k* con **validación cruzada** sobre el entrenamiento (sin mirar los datos de prueba) y nos quedamos con el mejor. Usamos valores impares para que la votación nunca empate.

In [ ]:
valores_k = list(range(1, 22, 2))
puntajes = [round(cross_val_score(KNeighborsClassifier(n_neighbors=k, algorithm="brute"),
                                  Xs_train, y_train, cv=5).mean(), 3) for k in valores_k]

# Si hay empate, nos quedamos con el k más pequeño (np.argmax toma el primero)
k = valores_k[int(np.argmax(puntajes))]

plt.figure(figsize=(6, 3))
plt.plot(valores_k, puntajes, marker="o")
plt.axvline(k, color="gray", linestyle="--")
plt.xlabel("k (número de vecinos)"); plt.ylabel("Accuracy (validación cruzada)")
plt.title(f"Mejor k = {k}")
plt.tight_layout(); plt.show()

In [ ]:
modelo = KNeighborsClassifier(n_neighbors=k, algorithm="brute")
modelo.fit(Xs_train, y_train)      # "entrenar" un KNN = guardar los datos

pred_test = modelo.predict(Xs_test)
exactitud = accuracy_score(y_test, pred_test)
precision = precision_score(y_test, pred_test)
recall = recall_score(y_test, pred_test)

print(f"k = {k}")
print(f"Accuracy:  {exactitud:.2%}")
print(f"Precisión: {precision:.2%}")
print(f"Recall:    {recall:.2%}")

ConfusionMatrixDisplay.from_predictions(y_test, pred_test, display_labels=["No llueve", "Sí llueve"],
                                        cmap="Blues", colorbar=False)
plt.title("Matriz de confusión (prueba)")
plt.show()

## 3. ¿Qué es un modelo KNN entrenado?

Aquí está la gran diferencia. Un KNN no tiene pesos que mostrar: `modelo.fit()` solo **guardó los datos**. Para clasificar un día nuevo, el servidor necesita:

1. Las **medias y desviaciones**, para normalizar el día nuevo igual que los de entrenamiento.
2. El valor de **k**.
3. **Todos los días de entrenamiento**, ya normalizados, con su etiqueta (llovió o no).

Y el algoritmo de predicción es:

$$d_j = \sqrt{\sum_{i} \left(z_i - z_{j,i}\right)^2} \qquad \text{(distancia del día nuevo a cada día } j\text{)}$$

$$p = \frac{\text{días con lluvia entre los } k \text{ más cercanos}}{k} \qquad \text{decisión} = \begin{cases} \text{Sí} & p \geq 0.5 \\ \text{No} & p < 0.5 \end{cases}$$

La distancia es la misma **norma euclidiana** de vectores que vimos en ERA 1.

In [ ]:
modelo_json = {
    "nombre": "KNN — ¿lloverá mañana?",
    "variables": variables,
    "media": media.tolist(),
    "desviacion": desviacion.tolist(),
    "k": int(k),
    "puntos": np.round(Xs_train, 4).tolist(),
    "etiquetas": y_train.astype(int).tolist(),
    "metricas_prueba": {
        "accuracy": round(float(exactitud), 4),
        "precision": round(float(precision), 4),
        "recall": round(float(recall), 4),
    },
    "clases": {"0": "No lloverá", "1": "Sí lloverá"},
}

Path("modelo.json").write_text(json.dumps(modelo_json, ensure_ascii=False), encoding="utf-8")

tamano = Path("modelo.json").stat().st_size
print(f"Días guardados: {len(modelo_json['puntos'])}")
print(f"Tamaño del archivo: {tamano:,} bytes")
print(f"La regresión logística ocupaba 435 bytes: este modelo es {tamano / 435:.0f} veces más grande")

> 💡 El tamaño del modelo **crece con los datos**. Con 225 días son unos KB; con un millón de días serían decenas de MB, y cada predicción tendría que calcular un millón de distancias. Es exactamente el tipo de problema que atacamos en el Módulo 11 con vectorización y paralelismo.

## 4. Verificar: ¿el JSON clasifica igual que scikit-learn?

La función de la app solo usa `math.dist`, que calcula la distancia euclidiana entre dos listas de números.

In [ ]:
def clasificar(datos, modelo):
    # 1. Normalizar el día nuevo con μ y σ del entrenamiento
    z = [(datos[v] - mu) / s for v, mu, s in zip(modelo["variables"], modelo["media"], modelo["desviacion"])]
    # 2. Distancia a cada día guardado
    distancias = sorted((math.dist(z, punto), j) for j, punto in enumerate(modelo["puntos"]))
    # 3. Votación de los k más cercanos
    vecinos = distancias[:modelo["k"]]
    votos = sum(modelo["etiquetas"][j] for _, j in vecinos)
    p = votos / modelo["k"]
    return p, int(p >= 0.5), vecinos


modelo_cargado = json.loads(Path("modelo.json").read_text())

resultados = [clasificar(fila, modelo_cargado) for fila in X_test.to_dict("records")]
p_json = np.array([p for p, _, _ in resultados])
clase_json = np.array([c for _, c, _ in resultados])

comparacion = pd.DataFrame({
    "p scikit-learn": modelo.predict_proba(Xs_test)[:, 1],
    "p JSON (Python puro)": p_json,
    "clase JSON": clase_json,
    "clase real": y_test.values,
}).round(4)

display(comparacion.head(6))
coinciden = (clase_json == modelo.predict(Xs_test)).mean()
print(f"Coincidencia en las clases de los {len(X_test)} días de prueba: {coinciden:.0%}")

Y lo mejor de KNN: podemos **ver por qué** decidió. Estos son los vecinos de un día de prueba, devueltos a sus unidades originales:

In [ ]:
dia = X_test.iloc[0].to_dict()
p, clase, vecinos = clasificar(dia, modelo_cargado)

tabla = pd.DataFrame(
    [[round(zj * s + mu, 1) for zj, mu, s in zip(modelo_cargado["puntos"][j], media, desviacion)]
     + [modelo_cargado["etiquetas"][j], round(d, 3)] for d, j in vecinos],
    columns=variables + ["llovió", "distancia"],
)
print("Día nuevo:", dia, f"→ p = {p:.2f} →", modelo_cargado["clases"][str(clase)])
tabla

## 5. Construir la app

Los mismos cuatro archivos. La novedad: la API también devuelve **los vecinos**, y la página los muestra en una tabla. Así el usuario ve en qué días parecidos se basó la decisión.

In [ ]:
carpeta_app = Path("app-lluvia-knn")
carpeta_app.mkdir(exist_ok=True)
shutil.copy("modelo.json", carpeta_app / "modelo.json")
print("Carpeta creada:", carpeta_app.resolve())

### 5.1 La API: `main.py`

In [ ]:
%%writefile app-lluvia-knn/main.py
"""
Módulo 12 — API del modelo KNN (¿lloverá mañana?)

Rutas:
    GET  /          → página con el formulario
    GET  /modelo    → información y métricas del modelo
    POST /predict   → votación de los k vecinos más cercanos
    GET  /docs      → documentación automática (la genera FastAPI)
"""
import json
import math
from pathlib import Path

from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field

CARPETA = Path(__file__).parent
MODELO = json.loads((CARPETA / "modelo.json").read_text(encoding="utf-8"))

app = FastAPI(
    title="¿Lloverá mañana? (KNN)",
    description="KNN con los días del Módulo 8, desplegado en el Módulo 12.",
    version="1.0",
)


class Dia(BaseModel):
    temperatura: float = Field(..., ge=-10, le=50, description="Temperatura promedio (°C)", examples=[24])
    humedad: float = Field(..., ge=0, le=100, description="Humedad relativa (%)", examples=[85])
    viento: float = Field(..., ge=0, le=150, description="Velocidad del viento (km/h)", examples=[12])


def a_unidades(punto):
    # De vuelta a °C, % y km/h:  x = z · σ + μ
    return {v: round(z * s + mu, 1)
            for v, z, mu, s in zip(MODELO["variables"], punto, MODELO["media"], MODELO["desviacion"])}


def clasificar(datos: dict):
    z = [(datos[v] - mu) / s for v, mu, s in zip(MODELO["variables"], MODELO["media"], MODELO["desviacion"])]
    distancias = sorted((math.dist(z, punto), j) for j, punto in enumerate(MODELO["puntos"]))
    vecinos = distancias[:MODELO["k"]]
    votos = sum(MODELO["etiquetas"][j] for _, j in vecinos)
    p = votos / MODELO["k"]
    return p, int(p >= 0.5), vecinos


@app.get("/", response_class=HTMLResponse, include_in_schema=False)
def pagina():
    return (CARPETA / "index.html").read_text(encoding="utf-8")


@app.get("/modelo")
def info_modelo():
    return {
        "nombre": MODELO["nombre"],
        "variables": MODELO["variables"],
        "k": MODELO["k"],
        "dias_guardados": len(MODELO["puntos"]),
        "metricas_prueba": MODELO["metricas_prueba"],
    }


@app.post("/predict")
def predict(dia: Dia):
    p, clase, vecinos = clasificar(dia.model_dump())
    return {
        "probabilidad": round(p, 4),
        "clase": clase,
        "decision": MODELO["clases"][str(clase)],
        "k": MODELO["k"],
        "vecinos": [
            {**a_unidades(MODELO["puntos"][j]), "llovio": MODELO["etiquetas"][j], "distancia": round(d, 3)}
            for d, j in vecinos
        ],
    }

### 5.2 La página: `index.html`

Los mismos deslizadores de la app de lluvia. El resultado muestra los votos de los vecinos (un ícono por vecino) y la tabla con los días más parecidos.

In [ ]:
%%writefile app-lluvia-knn/index.html
<!doctype html>
<html lang="es">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>¿Lloverá mañana? KNN</title>
  <style>
    :root { --fondo:#f2f5f4; --tarjeta:#fff; --texto:#1b2a2a; --suave:#5d6b6b; --acento:#227064; --borde:#d6dfdd;
            --lluvia:#2f5fa7; --seco:#c98a12; }
    * { box-sizing: border-box; }
    body { margin:0; font-family: system-ui, -apple-system, "Segoe UI", sans-serif; background:var(--fondo); color:var(--texto); }
    main { max-width: 640px; margin: 0 auto; padding: 32px 16px; }
    h1 { margin: 0 0 4px; font-size: 1.6rem; }
    .sub { color: var(--suave); margin: 0 0 24px; }
    .tarjeta { background: var(--tarjeta); border: 1px solid var(--borde); border-radius: 12px; padding: 24px; }
    .campo { margin-bottom: 22px; }
    .fila { display: flex; justify-content: space-between; align-items: baseline; margin-bottom: 8px; }
    label { font-weight: 600; }
    .valor { font-variant-numeric: tabular-nums; color: var(--acento); font-weight: 700; }
    input[type=range] { width: 100%; accent-color: var(--acento); }
    button { width: 100%; padding: 12px; font-size: 1rem; font-weight: 600; color: #fff;
             background: var(--acento); border: 0; border-radius: 8px; cursor: pointer; }
    button:disabled { opacity: .6; }
    #resultado { margin-top: 22px; display: none; }
    #decision { font-size: 1.8rem; font-weight: 700; margin-bottom: 8px; }
    #votos { font-size: 1.6rem; letter-spacing: 4px; margin: 4px 0 6px; }
    #detalle { color: var(--suave); font-size: .9rem; margin-bottom: 16px; }
    .tabla { overflow-x: auto; }
    table { width: 100%; border-collapse: collapse; font-size: .9rem; font-variant-numeric: tabular-nums; }
    th, td { padding: 6px 8px; text-align: right; border-bottom: 1px solid var(--borde); }
    th:first-child, td:first-child { text-align: left; }
    th { color: var(--suave); font-weight: 600; }
    footer { margin-top: 20px; font-size: .85rem; color: var(--suave); }
    a { color: var(--acento); }
  </style>
</head>
<body>
<main>
  <h1>🌦️ ¿Lloverá mañana? <small style="font-weight:400;color:var(--suave)">KNN</small></h1>
  <p class="sub">Busca los días más parecidos en el historial y les pide que voten</p>

  <form id="formulario" class="tarjeta">
    <div class="campo">
      <div class="fila"><label for="temperatura">Temperatura</label><span class="valor" id="v-temperatura"></span></div>
      <input id="temperatura" type="range" min="5" max="40" step="0.5" value="24" data-unidad=" °C">
    </div>
    <div class="campo">
      <div class="fila"><label for="humedad">Humedad</label><span class="valor" id="v-humedad"></span></div>
      <input id="humedad" type="range" min="10" max="100" step="1" value="85" data-unidad=" %">
    </div>
    <div class="campo">
      <div class="fila"><label for="viento">Viento</label><span class="valor" id="v-viento"></span></div>
      <input id="viento" type="range" min="0" max="40" step="1" value="12" data-unidad=" km/h">
    </div>

    <button id="boton" type="submit">Preguntar a los vecinos</button>

    <div id="resultado">
      <div id="decision"></div>
      <div id="votos"></div>
      <div id="detalle"></div>
      <div class="tabla">
        <table>
          <thead><tr><th>Vecino</th><th>Temp. (°C)</th><th>Humedad (%)</th><th>Viento (km/h)</th><th>¿Llovió?</th><th>Distancia</th></tr></thead>
          <tbody id="filas"></tbody>
        </table>
      </div>
    </div>
  </form>

  <footer>
    <span id="info-modelo">Cargando información del modelo…</span> ·
    <a href="/docs">Ver la API (/docs)</a>
  </footer>
</main>

<script>
  const campos = ["temperatura", "humedad", "viento"];

  for (const c of campos) {
    const control = document.getElementById(c);
    const mostrar = () => document.getElementById("v-" + c).textContent = control.value + control.dataset.unidad;
    control.addEventListener("input", mostrar);
    mostrar();
  }

  fetch("/modelo").then(r => r.json()).then(m => {
    const k = m.metricas_prueba;
    document.getElementById("info-modelo").textContent =
      `k = ${m.k} · ${m.dias_guardados} días guardados · Accuracy ${(k.accuracy * 100).toFixed(0)} % · Recall ${(k.recall * 100).toFixed(0)} %`;
  });

  document.getElementById("formulario").addEventListener("submit", async (evento) => {
    evento.preventDefault();
    const boton = document.getElementById("boton");
    boton.disabled = true;

    // 1. Reunir los datos
    const datos = {};
    for (const c of campos) datos[c] = Number(document.getElementById(c).value);

    // 2. Enviarlos a la API
    try {
      const respuesta = await fetch("/predict", {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify(datos),
      });
      if (!respuesta.ok) throw new Error(`Error ${respuesta.status}`);
      const r = await respuesta.json();

      // 3. Mostrar el resultado
      const lluvias = r.vecinos.filter(v => v.llovio === 1).length;
      const decision = document.getElementById("decision");
      decision.textContent = (r.clase === 1 ? "☔ " : "☀️ ") + r.decision;
      decision.style.color = r.clase === 1 ? "var(--lluvia)" : "var(--seco)";
      document.getElementById("votos").textContent = r.vecinos.map(v => v.llovio === 1 ? "☔" : "☀️").join("");
      document.getElementById("detalle").textContent =
        `${lluvias} de ${r.k} días parecidos tuvieron lluvia · probabilidad ${(r.probabilidad * 100).toFixed(0)} %`;
      document.getElementById("filas").innerHTML = r.vecinos.map((v, i) =>
        `<tr><td>${i + 1}</td><td>${v.temperatura.toFixed(1)}</td><td>${v.humedad.toFixed(1)}</td><td>${v.viento.toFixed(1)}</td>` +
        `<td>${v.llovio === 1 ? "☔ Sí" : "☀️ No"}</td><td>${v.distancia}</td></tr>`).join("");
    } catch (e) {
      document.getElementById("decision").textContent = "No se pudo calcular";
      document.getElementById("detalle").textContent = e.message;
    }
    document.getElementById("resultado").style.display = "block";
    boton.disabled = false;
  });
</script>
</body>
</html>

### 5.3 Dependencias y README

In [ ]:
%%writefile app-lluvia-knn/requirements.txt
fastapi
uvicorn

In [ ]:
%%writefile app-lluvia-knn/README.md
# ¿Lloverá mañana? (KNN) — Módulo 12

KNN con los días simulados del Módulo 8, desplegado como API con FastAPI en Vercel.

## Enlaces
- App: https://TU-PROYECTO.vercel.app
- Documentación de la API: https://TU-PROYECTO.vercel.app/docs

## Archivos
| Archivo | Rol |
|---|---|
| `modelo.json` | μ, σ, k y los días de entrenamiento con su etiqueta |
| `main.py` | API: busca los k vecinos y los pone a votar en `/predict` |
| `index.html` | Página con los deslizadores y la tabla de vecinos |
| `requirements.txt` | Dependencias del servidor |

## Reflexión
(Escribe aquí tus respuestas a los retos del notebook)

## 6. Probar la app antes de desplegarla

In [ ]:
import sys, importlib
sys.path.insert(0, str(carpeta_app.resolve()))
import main
importlib.reload(main)

from fastapi.testclient import TestClient
cliente = TestClient(main.app)

print("GET /        →", cliente.get("/").status_code)
print("GET /modelo  →", cliente.get("/modelo").json())

r = cliente.post("/predict", json={"temperatura": 24, "humedad": 85, "viento": 12}).json()
print("POST /predict →", {c: r[c] for c in ("probabilidad", "clase", "decision", "k")})
pd.DataFrame(r["vecinos"])

In [ ]:
r = cliente.post("/predict", json={"temperatura": 24, "humedad": 150, "viento": 12})
print("Humedad 150 % → código", r.status_code, "|", r.json()["detail"][0]["msg"])

### (Opcional) Ver la página dentro de Colab

In [ ]:
import subprocess, time

servidor = subprocess.Popen(["uvicorn", "main:app", "--port", "8002"], cwd=carpeta_app)
time.sleep(3)

try:
    from google.colab import output
    output.serve_kernel_port_as_window(8002)
except ImportError:
    print("No estás en Colab: abre http://localhost:8002 en tu navegador")

## 7. Descargar los archivos

In [ ]:
# __pycache__ la crea Python al importar main.py en las pruebas: no se sube a GitHub
shutil.rmtree(carpeta_app / "__pycache__", ignore_errors=True)

shutil.make_archive("app-lluvia-knn", "zip", carpeta_app)
import zipfile
print("Contenido del zip:", sorted(zipfile.ZipFile(f"{carpeta_app}.zip").namelist()))

try:
    from google.colab import files
    files.download("app-lluvia-knn.zip")
except ImportError:
    print("Archivo listo: app-lluvia-knn.zip")

## 8. Desplegar en Vercel

El mismo procedimiento, en un **repositorio nuevo**:

1. **GitHub:** crea un repositorio público llamado `app-lluvia-knn` y sube los **5 archivos** del zip a la raíz.
2. **Vercel:** *Add New… → Project*. Si el repositorio no aparece, usa *Adjust GitHub App Permissions* para darle acceso.
3. Presiona **Import**, verifica que el *Framework Preset* diga **FastAPI** y presiona **Deploy**.
4. Prueba la URL: `/`, `/docs` y `/modelo`.

> 💡 En GitHub, abre `modelo.json`: verás una sola línea muy larga con todos los días. Compárala con el JSON de la regresión logística, que tenía tres pesos.

## 9. Retos

Responde en el `README.md` de tu repositorio:

1. **Cambia k sin reentrenar.** En GitHub, edita `modelo.json` y cambia `"k": ` por `1` y luego por `21`. Prueba el mismo día (24 °C, 85 %, 12 km/h) con cada valor. ¿Qué pasa con la decisión y con la tabla de vecinos? ¿Por qué en KNN se puede cambiar k sin volver a entrenar?
2. **Lee la tabla de vecinos.** ¿Los días más parecidos se parecen sobre todo en humedad, en temperatura o en viento? ¿Por qué crees que es así?
3. **Compara con la regresión logística.** Prueba el mismo día en tu app de regresión logística y en esta. ¿Coinciden? ¿Cuál de las dos te explica mejor su decisión?
4. **Piensa en escala.** Este modelo guarda 225 días. ¿Qué pasaría con el tamaño de `modelo.json` y con el tiempo de cada predicción si guardara 10 años de datos de una estación que mide cada minuto? ¿Qué técnicas del Módulo 11 ayudarían?
5. **Privacidad.** El JSON de KNN contiene los datos de entrenamiento y cualquiera puede verlos en GitHub. ¿Sería un problema si los datos fueran de pacientes o clientes? ¿Pasa lo mismo con la regresión logística?

## Cierre

Con tres modelos distintos, el despliegue fue el mismo procedimiento: **exportar lo que el modelo necesita para predecir, envolverlo en una API y publicarlo**. Lo que cambió fue qué significa "lo que el modelo necesita":

| Modelo | El JSON guarda | Tamaño aproximado |
|---|---|---|
| Regresión lineal | μ, σ, 8 pesos y un sesgo | ~1 KB |
| Regresión logística | 3 pesos, un sesgo y el umbral | ~0.4 KB |
| KNN | μ, σ, k y todos los días de entrenamiento | Decenas de KB, y crece con los datos |

**Próxima sesión:** empaquetaremos estas apps en contenedores Docker.